# Episode 7 — Control panels with ipywidgets

**ipywidgets** puts sliders, buttons and text boxes in a notebook. Each widget has a `value`;
when the value changes, Python calls a function you choose (a **callback**). Wire the
callbacks to the board and you have a control panel for your hardware.

| Widget | Use it for |
| --- | --- |
| `IntSlider`, `FloatSlider` | a number in a range |
| `ToggleButton` | on / off |
| `Button` | an action (`on_click`) |
| `Text` | a short string |
| `HTML`, `Label` | showing values |
| `HBox`, `VBox` | laying widgets out in rows and columns |

In [ ]:
from pynq.overlays.base import BaseOverlay
import ipywidgets as widgets
from IPython.display import display

base = BaseOverlay("base.bit")

## 1. The quickest way: `interact`

`interact` builds a widget from the argument range and calls the function on every change.
Drag the slider and watch the LEDs.

In [ ]:
@widgets.interact(value=(0, 15))
def set_leds(value=5):
    base.leds[0:4].write(value)

## 2. A toggle button per LED

`observe` calls the function with the change; `change["new"]` is the new value.

In [ ]:
def make_led_button(i):
    button = widgets.ToggleButton(description="LED %d" % i, value=False)
    def changed(change):
        if change["new"]:
            base.leds[i].on()
        else:
            base.leds[i].off()
    button.observe(changed, names="value")
    return button

led_buttons = [make_led_button(i) for i in range(4)]
display(widgets.HBox(led_buttons))

## 3. A buzzer with a frequency slider

Fit the J10 jumper (pins 1–2).

In [ ]:
freq = widgets.IntSlider(description="Hz", min=200, max=4000, step=50, value=1000)
sound = widgets.ToggleButton(description="Sound", value=False)

def update_buzzer(change=None):
    if sound.value:
        base.buzzer.tone(freq.value)
    else:
        base.buzzer.off()

freq.observe(update_buzzer, names="value")
sound.observe(update_buzzer, names="value")
display(widgets.HBox([sound, freq]))

## 4. Type on the 7-segment display

In [ ]:
message = widgets.Text(description="Display", value="HI", placeholder="up to 4 characters")

def show_message(change):
    base.seven_seg.show_text(change["new"][:4])

message.observe(show_message, names="value")
display(message)
base.seven_seg.show_text(message.value)

## 5. A live status panel

An asyncio task reads the board twice a second and updates an `HTML` widget, while the
notebook stays free for other cells. **Stop** cancels the task.

In [ ]:
import asyncio

status = widgets.HTML()
stop = widgets.Button(description="Stop", button_style="danger")

async def update_status():
    while True:
        sw = base.switches.read()
        btn = base.buttons.read()
        status.value = (
            "<b>Switches</b> {:04b} &nbsp; <b>Buttons</b> {:02b} &nbsp; "
            "<b>Chip</b> {:.1f} °C &nbsp; <b>LM35</b> {:.1f} °C &nbsp; <b>Light</b> {:.2f} V"
        ).format(sw, btn, base.xadc.temperature(), base.xadc.lm35_celsius(),
                 base.xadc.light_level())
        await asyncio.sleep(0.5)

task = asyncio.ensure_future(update_status())
stop.on_click(lambda b: task.cancel())
display(widgets.VBox([status, stop]))

## 6. Everything in one panel

The same widgets, arranged with `VBox` and `HBox`. A widget can be shown in more than
one place; all copies stay in step.

In [ ]:
panel = widgets.VBox([
    widgets.HTML("<h3>EDGE ZYNQ control panel</h3>"),
    widgets.HBox(led_buttons),
    widgets.HBox([sound, freq]),
    message,
    status,
    stop,
])
display(panel)

## Try it yourself

Add a **Dropdown** with the choices "off", "blink" and "chase" that sets an LED pattern,
using an asyncio task like the status panel's.

In [ ]:
# your code here